# 🛩️ Example: Working with the LAPAN Surveillance Aircraft Model (LSU-05 NG)

This notebook demonstrates the usage of the linear longitudinal model of the LAPAN LSU-05 NG surveillance aircraft in the TensorAeroSpace environment.

## 📋 What we will do:
1. Import the required libraries
2. Configure time parameters and the reference signal
3. Create and initialize the environment
4. Execute one simulation step
5. Analyze the results

## 📚 Importing Libraries

Loading all necessary modules for working with the aircraft model:

In [1]:
# Основные библиотеки
import gymnasium as gym
import numpy as np

# Импорт TensorAeroSpace для регистрации окружений
import tensoraerospace

# Модули TensorAeroSpace
from tensoraerospace.utils import generate_time_period, convert_tp_to_sec_tp
from tensoraerospace.signals.standart import unit_step

## ⚙️ Simulation Parameters Setup

Defining time parameters and creating a reference signal for pitch angle control:

In [2]:
# Параметры дискретизации
dt = 0.01  # Шаг дискретизации (секунды)

# Генерация временного периода
tp = generate_time_period(tn=20, dt=dt)  # 20 секунд симуляции
tps = convert_tp_to_sec_tp(tp, dt=dt)    # Преобразование в секунды
number_time_steps = len(tp)             # Общее количество временных шагов

# Создание ступенчатого опорного сигнала для угла тангажа (theta)
reference_signals = np.reshape(
    unit_step(degree=5, tp=tp, time_step=10, output_rad=True), 
    [1, -1]
)

print(f"📊 Параметры симуляции:")
print(f"   • Время симуляции: {tp[-1]:.1f} сек")
print(f"   • Шаг дискретизации: {dt} сек")
print(f"   • Количество шагов: {number_time_steps}")
print(f"   • Форма опорного сигнала: {reference_signals.shape}")

📊 Параметры симуляции:
   • Время симуляции: 2002.0 сек
   • Шаг дискретизации: 0.01 сек
   • Количество шагов: 2002
   • Форма опорного сигнала: (1, 2002)


## 🚀 Creating and Initializing the Environment

Creating the LAPAN LSU-05 NG environment with the specified parameters and performing a reset:

In [ ]:
# Создание среды LAPAN LSU-05 NG
env = gym.make(
    "LinearLongitudinalLAPAN-v0",
    number_time_steps=number_time_steps,
    initial_state=[[0], [0], [0], [0]],  # Начальное состояние [u, w, q, theta]
    reference_signal=reference_signals,
    tracking_states=["theta"]
)

# Инициализация среды
state, info = env.reset()

print(f"✅ Среда успешно создана и инициализирована!")
print(f"📈 Начальное состояние: {state.flatten()}")
print(f"🎯 Форма пространства состояний: {env.observation_space.shape}")
print(f"🎮 Форма пространства действий: {env.action_space.shape}")
print(f"📊 Отслеживаемые состояния: {env.unwrapped.tracking_states}")
print(f"📋 Пространство состояний: {env.unwrapped.state_space}")
print(f"📤 Пространство выходов: {env.unwrapped.output_space}")

✅ Среда успешно создана и инициализирована!
📈 Начальное состояние: [0. 0. 0. 0.]
🎯 Форма пространства состояний: (2, 1)
🎮 Форма пространства действий: (1, 1)
📊 Отслеживаемые состояния: ['theta']
📋 Пространство состояний: ['theta', 'q']
📤 Пространство выходов: ['theta', 'q']


/Users/asmazaev/TensorAeroSpace-Origin/.venv/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:159: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


## 🎮 Executing a Simulation Step

Applying a control action and observing the system response:

In [ ]:
# Применение управляющего воздействия
control_input = np.array([1.0], dtype=np.float32)

# Выполнение одного шага симуляции
state, reward, terminated, truncated, info = env.step(control_input)

print(f"🎯 Управляющее воздействие: {control_input[0]:.2f} градусов")
print(f"📊 Новое состояние: {state.flatten()}")
theta_idx = env.unwrapped.state_space.index("theta")
q_idx = env.unwrapped.state_space.index("q")
print(f"   • Угол тангажа (theta): {np.rad2deg(state[theta_idx, 0]):.4f} град")
print(f"   • Угловая скорость (q): {np.rad2deg(state[q_idx, 0]):.4f} град/с")
# Обработка награды (может быть массивом или скаляром)
reward_value = reward[0] if isinstance(reward, np.ndarray) else reward
print(f"🏆 Награда: {reward_value:.6f}")
print(f"🔚 Завершено: {terminated}")
print(f"⏰ Прервано: {truncated}")

🎯 Управляющее воздействие: 1.00 градусов
📊 Новое состояние: [ 0.01291399 -0.90360685 -1.21675473 -0.00625333]
   • Угол тангажа (theta): 0.7399 град
   • Угловая скорость (q): -51.7729 град/с
🏆 Награда: 0.012914
🔚 Завершено: False
⏰ Прервано: False


/Users/asmazaev/TensorAeroSpace-Origin/.venv/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:135: UserWarning: WARN: The obs returned by the `step()` method was expecting numpy array dtype to be float32, actual type: float64
  logger.warn(
/Users/asmazaev/TensorAeroSpace-Origin/.venv/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:159: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
/Users/asmazaev/TensorAeroSpace-Origin/.venv/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:246: UserWarning: WARN: The reward returned by `step()` must be a float, int, np.integer or np.floating, actual type: <class 'numpy.ndarray'>
  logger.warn(
